In [10]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import  *
import time

In [2]:
spark = SparkSession.builder \
.appName("DataframeWriter Avro") \
.master("local[2]") \
.config("spark.jars.packages","org.apache.spark:spark-avro_2.12:3.4.1") \
.getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-avro_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a4b3675d-488e-4971-a2bb-9e415fde7d0d;1.0
	confs: [default]
	found org.apache.spark#spark-avro_2.12;3.4.1 in central
	found org.tukaani#xz;1.9 in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spark-avro_2.12/3.4.1/spark-avro_2.12-3.4.1.jar ...
	[SUCCESSFUL ] org.apache.spark#spark-avro_2.12;3.4.1!spark-avro_2.12.jar (239ms)
downloading https://repo1.maven.org/maven2/org/tukaani/xz/1.9/xz-1.9.jar ...
	[SUCCESSFUL ] org.tukaani#xz;1.9!xz.jar (191ms)
:: resolution report :: resolve 5053ms :: artifacts dl 445ms
	:: modules in use:
	org.apache.spark#spark-avro_2.12;3.4.1 from central in [default]
	org.tukaani#xz;1.9 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules    

In [3]:
df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.option("compression","gzip") \
.load("file:///opt/examples/datasets/Hotel_Reviews.csv.gz")

In [4]:
df2 = df.withColumn("Tags", 
                     F.split(F.col("Tags"), ",")
                     .cast(ArrayType(StringType()))) \
.withColumn("Review_Date", F.to_date(F.col("Review_Date"),"M/d/yyyy"))

In [8]:
df2.limit(2).toPandas()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,2017-08-03,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[[' Leisure trip ', ' Couple ', ' Duplex Dou...",0 days,52.3605759,4.9159683
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,2017-08-03,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[[' Leisure trip ', ' Couple ', ' Duplex Dou...",0 days,52.3605759,4.9159683


In [9]:
df2.printSchema()

root
 |-- Hotel_Address: string (nullable = true)
 |-- Additional_Number_of_Scoring: integer (nullable = true)
 |-- Review_Date: date (nullable = true)
 |-- Average_Score: double (nullable = true)
 |-- Hotel_Name: string (nullable = true)
 |-- Reviewer_Nationality: string (nullable = true)
 |-- Negative_Review: string (nullable = true)
 |-- Review_Total_Negative_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews: integer (nullable = true)
 |-- Positive_Review: string (nullable = true)
 |-- Review_Total_Positive_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews_Reviewer_Has_Given: integer (nullable = true)
 |-- Reviewer_Score: double (nullable = true)
 |-- Tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- days_since_review: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lng: string (nullable = true)



# avro

In [11]:

start_time = time.time()

df2.write \
.format("avro") \
.mode("overwrite") \
.save("file:///tmp/pyspark_output_data/hotel_review_avro")


print("----------- %s secs ----------" %(time.time() - start_time))

----------- 22.092730283737183 secs ----------


In [12]:
! ls -lh /tmp/pyspark_output_data/hotel_review_avro

total 78M
-rw-r--r--. 1 root root   0 Sep 28 18:58 _SUCCESS
-rw-r--r--. 1 root root 78M Sep 28 18:58 part-00000-821eb81d-9aac-4fb1-905d-86a463252ef7-c000.avro


# avro with compression

In [13]:
start_time = time.time()

df2.write \
.format("avro") \
.mode("overwrite") \
.option("compression","bzip2") \
.save("file:///tmp/pyspark_output_data/hotel_review_avro_compress")


print("----------- %s secs ----------" %(time.time() - start_time))

----------- 97.95378136634827 secs ----------


In [14]:
! ls -lh /tmp/pyspark_output_data/hotel_review_avro_compress

total 42M
-rw-r--r--. 1 root root   0 Sep 28 19:00 _SUCCESS
-rw-r--r--. 1 root root 42M Sep 28 19:00 part-00000-cceee94f-fc0d-4117-8441-50e02237816c-c000.avro


In [15]:
spark.stop()

23/09/28 19:09:17 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-116e1a7b-5494-4096-8bb0-314051389a95. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-116e1a7b-5494-4096-8bb0-314051389a95
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:177)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:113)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1231)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:65)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:62)
	at scala.collection.IndexedSeqOptimized.foreach(IndexedSeqOptimized.scala:36)
	at scala.collection.IndexedSeqOptimized.foreach$(IndexedSeqOptimized.scala:33)
	at scala.collection.mutable.ArrayOps$ofRef.foreach(Array